In [29]:
import os
import pickle
import pandas as pd
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit

In [30]:
def print_split_stats(cohort, splits):
    for split_name, ids in splits.items():
        y = cohort.loc[cohort['ts_id'].astype(str).isin(ids), 'in_hospital_mortality']

        print(f"{split_name}: n={len(y)}")

        # binary stats
        if y.nunique() <= 2:
            pos_rate = y.mean()
            print(f"  positives: {y.sum()}, rate: {pos_rate:.3f}")
        # multiclass stats
        else:
            counts = y.value_counts()
            for cls, cnt in counts.items():
                print(f"  class {cls}: {cnt} ({cnt/len(y):.3f})")


def compute_splits_stf(dataset, data_paths, OUT_PATH=None, n_splits=3, train_frac=0.8, seed=123):

    # File paths
    filepath = data_paths[dataset]
    if OUT_PATH is None:
        OUT_PATH = os.path.join(os.path.dirname(filepath), "folds")
    os.makedirs(OUT_PATH, exist_ok=True)

    # Load dataset
    with open(filepath, "rb") as f:
        ts, oc, train_ids, val_ids, test_ids = pickle.load(f)

    X = oc['ts_id']
    y = oc['in_hospital_mortality']

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    for i, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        test_ids = X.iloc[test_idx].astype(str).values
        train_valid_ids = X.iloc[train_idx].astype(str).values
        y_train_valid = y.iloc[train_idx]

        # Stratified 80/20 split for train/val
        sss = StratifiedShuffleSplit(n_splits=1, train_size=train_frac, random_state=seed)
        train_idx_inner, val_idx_inner = next(sss.split(train_valid_ids, y_train_valid))
        train_ids_fold = train_valid_ids[train_idx_inner]
        val_ids_fold   = train_valid_ids[val_idx_inner]

        # -----------------------------
        # Check for overlap
        # -----------------------------
        overlap_train_val = set(train_ids_fold) & set(val_ids_fold)
        overlap_train_test = set(train_ids_fold) & set(test_ids)
        overlap_val_test = set(val_ids_fold) & set(test_ids)

        if overlap_train_val or overlap_train_test or overlap_val_test:
            raise ValueError(
                f"Overlap detected in fold {i}!\n"
                f"train/val: {overlap_train_val}\n"
                f"train/test: {overlap_train_test}\n"
                f"val/test: {overlap_val_test}"
            )

        # Print stats
        print(f"############ FOLD {i} ############")
        print_split_stats(oc, {"Train": train_ids_fold, "Val": val_ids_fold, "Test": test_ids})

        # Save fold
        out_file = os.path.join(OUT_PATH, f'{dataset}_fold_{i}.pkl')
        with open(out_file, 'wb') as f:
            pickle.dump([ts, oc, train_ids_fold, val_ids_fold, test_ids], f)
        print(f"Saved fold {i} to {out_file}\n")

In [31]:
data_paths = {
    "physionet_2012" : "/home/christel-sirocchi/GitHub/DATASETS/Physionet_2012/preprocessed_strats/physionet_2012.pkl",
    "mimic_iii" : "/home/christel-sirocchi/GitHub/DATASETS/MIMIC_III/preprocessed_strats/mimic_iii.pkl"
}

In [32]:
# Example usage
compute_splits_stf("physionet_2012", data_paths)

############ FOLD 0 ############
Train: n=6393
  positives: 910, rate: 0.142
Val: n=1599
  positives: 228, rate: 0.143
Test: n=3996
  positives: 569, rate: 0.142
Saved fold 0 to /home/christel-sirocchi/GitHub/DATASETS/Physionet_2012/preprocessed_strats/folds/physionet_2012_fold_0.pkl

############ FOLD 1 ############
Train: n=6393
  positives: 910, rate: 0.142
Val: n=1599
  positives: 228, rate: 0.143
Test: n=3996
  positives: 569, rate: 0.142
Saved fold 1 to /home/christel-sirocchi/GitHub/DATASETS/Physionet_2012/preprocessed_strats/folds/physionet_2012_fold_1.pkl

############ FOLD 2 ############
Train: n=6393
  positives: 910, rate: 0.142
Val: n=1599
  positives: 228, rate: 0.143
Test: n=3996
  positives: 569, rate: 0.142
Saved fold 2 to /home/christel-sirocchi/GitHub/DATASETS/Physionet_2012/preprocessed_strats/folds/physionet_2012_fold_2.pkl



In [33]:
compute_splits_stf("mimic_iii", data_paths)

############ FOLD 0 ############
Train: n=28197
  positives: 3470, rate: 0.123
Val: n=7050
  positives: 867, rate: 0.123
Test: n=17624
  positives: 2169, rate: 0.123
Saved fold 0 to /home/christel-sirocchi/GitHub/DATASETS/MIMIC_III/preprocessed_strats/folds/mimic_iii_fold_0.pkl

############ FOLD 1 ############
Train: n=28197
  positives: 3470, rate: 0.123
Val: n=7050
  positives: 867, rate: 0.123
Test: n=17624
  positives: 2169, rate: 0.123
Saved fold 1 to /home/christel-sirocchi/GitHub/DATASETS/MIMIC_III/preprocessed_strats/folds/mimic_iii_fold_1.pkl

############ FOLD 2 ############
Train: n=28198
  positives: 3470, rate: 0.123
Val: n=7050
  positives: 868, rate: 0.123
Test: n=17623
  positives: 2168, rate: 0.123
Saved fold 2 to /home/christel-sirocchi/GitHub/DATASETS/MIMIC_III/preprocessed_strats/folds/mimic_iii_fold_2.pkl

